# Part 2: Batch Process - OME-Zarr Pipeline

**A modernized cycle processing pipeline using OME-Zarr format throughout:**

- **OME-Zarr storage** - Chunked, parallel I/O for all stages
- **Dask integration** - Lazy loading and parallel processing
- **GPU acceleration** - CuPy for compute-intensive operations
- **Multi-GPU ready** - dask-cuda for distributed processing
- **No file size limits** - Unlike TIFF 4GB limit
- **Cloud-ready** - OME-NGFF compliant format

## Pipeline Stages

```
Raw Tiles → BaSiC Correction → Stitching → Deconvolution → EDF → Registration
   ↓              ↓               ↓            ↓           ↓         ↓
 .zarr         .zarr          .zarr        .zarr       .zarr    .ome.zarr
```

---

## 1. Import Packages and Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# Core imports
import os
import sys
import gc
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path
from datetime import datetime
from itertools import chain, repeat
from tqdm.notebook import tqdm
import warnings
import platform

warnings.filterwarnings("ignore")

# Zarr and Dask imports
import zarr
import dask
import dask.array as da
from dask.distributed import Client, LocalCluster

# Image processing
from scipy import ndimage
from skimage import morphology
from skimage.io import imread, imsave
from skimage.io.collection import alphanumeric_key

# Local modules
from Kstitch.stitching import stitch_images

# Add src to path for kintsugi modules
base_dir = "C:\\Users\\smith6jt"
sys.path.insert(0, os.path.join(base_dir, 'KINTSUGI', 'src'))

from kintsugi.zarr_io import KintsugiZarr, DEFAULT_COMPRESSOR
from kintsugi.edf import EDFProcessor
from kintsugi.kcorrect_gpu import KCorrectGPU, KCorrectGPUFunc, check_gpu

# Check GPU for BaSiC
gpu_available, gpu_msg = check_gpu()
print(f"GPU BaSiC: {gpu_msg}")

os_system = platform.system()
print(f"Notebook started: {datetime.now()}")
print(f"Platform: {os_system}")

In [ ]:
# Check GPU availability
print("\n" + "="*50)
print("Hardware Detection")
print("="*50)

try:
    import cupy as cp
    gpu_mem = cp.cuda.Device().mem_info
    print(f"✓ CuPy GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
    print(f"  Memory: {gpu_mem[1]/1e9:.1f} GB total, {gpu_mem[0]/1e9:.1f} GB free")
    USE_GPU = True
except Exception as e:
    print(f"✗ CuPy GPU: Not available ({e})")
    USE_GPU = False

try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"✓ System RAM: {mem.total/1e9:.1f} GB total, {mem.available/1e9:.1f} GB available")
    print(f"✓ CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")
except ImportError:
    print("  psutil not installed - RAM info unavailable")

print("="*50)

## 2. Configure Paths and Parameters

In [ ]:
# ============ DIRECTORY PATHS ============
base_dir = "C:\\Users\\smith6jt"
data_name = "2008CC2B"

image_dir = os.path.join(base_dir, 'KINTSUGI', 'data', f'{data_name}_raw')
zarr_path = os.path.join(base_dir, 'KINTSUGI', 'data', f'{data_name}.zarr')

print(f"Raw image directory: {image_dir}")
print(f"Zarr store: {zarr_path}")

In [ ]:
# ============ PROCESSING PARAMETERS ============
# Grid configuration
n_rows = 9              # Number of tile rows
n_cols = 7              # Number of tile columns

# Processing range
start_cycle = 1
end_cycle = 1
start_channel = 1
end_channel = 4
n_zplanes = 17          # Total z-planes

# Stitching parameters
pou = 0.5               # Percent overlap uncertainty
overlap_percentage = 30 # Expected tile overlap %

# Compute settings
n_workers = 8           # Dask workers
use_gpu = USE_GPU       # GPU acceleration
smooth_borders = True   # Border smoothing

# BaSiC correction parameters
BASIC_IF_DARKFIELD = True
BASIC_MAX_ITERATIONS = 500
BASIC_OPTIMIZATION_TOLERANCE = 1e-6
BASIC_MAX_REWEIGHT_ITERATIONS = 25
BASIC_REWEIGHT_TOLERANCE = 1e-3

# Zarr chunking (optimized for GPU memory and parallel I/O)
ZARR_CHUNKS = (1024, 1024)  # Chunk size for 2D arrays

# Channel naming
channel_name_dict = {
    1: ["DAPI", "Blank1a", "Blank1b", "Blank1c"],
    2: ["DAPI", "CD31", "CD8", "Empty2c"],
    3: ["DAPI", "CD20", "Ki67", "CD3e"],
    4: ["DAPI", "SMActin", "Podoplanin", "CD68"],
    5: ["DAPI", "PanCK", "CD21", "CD4"],
    6: ["DAPI", "Lyve1", "CD45RO", "CD11c"],
    7: ["DAPI", "CD35", "ECAD", "CD107a"],
    8: ["DAPI", "CD34", "CD44", "HLADR"],
    9: ["DAPI", "Empty9a", "FoxP3", "CD163"],
    10: ["DAPI", "Empty10a", "CollagenIV", "Vimentin"],
    11: ["DAPI", "Empty11a", "CD15", "CD45"],
    12: ["DAPI", "Empty12a", "CD5", "CD1c"],
    13: ["DAPI", "Blank13a", "Blank13b", "Blank13c"]
}

# Build tile coordinate lists (snake pattern)
rows = list(chain.from_iterable(repeat(row, n_cols) for row in range(n_rows)))
cols = list(chain.from_iterable(range(n_cols) if row % 2 == 0 else range(n_cols - 1, -1, -1) for row in range(n_rows)))

print(f"\nConfiguration:")
print(f"  Grid: {n_rows} x {n_cols} = {n_rows * n_cols} tiles")
print(f"  Cycles: {start_cycle} to {end_cycle}")
print(f"  Channels: {start_channel} to {end_channel}")
print(f"  Z-planes: {n_zplanes}")
print(f"  GPU: {use_gpu}")

## 3. Initialize Zarr Store

In [ ]:
# Create or open zarr store
store = KintsugiZarr(zarr_path, mode='a')

# Initialize cycles
for cycle in range(start_cycle, end_cycle + 1):
    store.create_cycle(
        cycle=cycle,
        n_channels=end_channel - start_channel + 1,
        n_zplanes=n_zplanes,
        channel_names=channel_name_dict.get(cycle)
    )
    print(f"Initialized cycle {cycle} in zarr store")

print(f"\nZarr store ready at: {zarr_path}")

## 4. Start Dask Cluster

For multi-GPU support, replace `LocalCluster` with `dask_cuda.LocalCUDACluster`

In [ ]:
# Cluster mode selection
# Options: 'local' (CPU workers), 'gpu' (single GPU), 'multi_gpu' (dask-cuda multi-GPU)
CLUSTER_MODE = 'local'

if CLUSTER_MODE == 'multi_gpu':
    # Multi-GPU cluster using dask-cuda
    try:
        from dask_cuda import LocalCUDACluster
        cluster = LocalCUDACluster(
            n_workers=None,  # One worker per GPU
            threads_per_worker=1,
            memory_limit='auto',  # Auto-detect GPU memory
            device_memory_limit='auto',
            rmm_pool_size='14GB',  # RMM memory pool for faster allocations
            enable_nvlink=True  # Enable NVLink if available
        )
        print("Multi-GPU cluster initialized")
    except ImportError:
        print("dask-cuda not available, falling back to LocalCluster")
        cluster = LocalCluster(n_workers=n_workers, threads_per_worker=2, memory_limit='8GB')
    except Exception as e:
        print(f"Error creating CUDA cluster: {e}")
        print("Falling back to LocalCluster")
        cluster = LocalCluster(n_workers=n_workers, threads_per_worker=2, memory_limit='8GB')

elif CLUSTER_MODE == 'gpu':
    # Single GPU mode - reduce workers to avoid GPU memory contention
    cluster = LocalCluster(
        n_workers=2,  # Fewer workers for GPU mode
        threads_per_worker=1,
        memory_limit='16GB'
    )
    print("Single GPU cluster initialized (reduced workers for GPU memory)")

else:
    # Standard CPU cluster
    cluster = LocalCluster(
        n_workers=n_workers,
        threads_per_worker=2,
        memory_limit='8GB'
    )
    print("CPU cluster initialized")

client = Client(cluster)

print(f"Dask dashboard: {client.dashboard_link}")
print(f"Workers: {len(client.scheduler_info()['workers'])}")
client

---
## 5. Illumination Correction and Stitching

Process tiles with BaSiC correction and stitch into mosaics.
Results are written directly to zarr.

In [ ]:
def smooth_tile_borders_2d(
    stitched_plane: np.ndarray,
    result_df,
    tile_shape: tuple,
    border_width: int = 5,
    median_size: int = 7,
    sigma: float = 1.0
) -> np.ndarray:
    """
    Fast boundary smoothing - only processes seam regions.
    """
    H, W = stitched_plane.shape
    tile_h, tile_w = map(int, tile_shape)

    # Build seam mask
    seam_mask = np.zeros((H, W), dtype=bool)
    for x0, y0 in zip(result_df["x_pos2"].astype(int), result_df["y_pos2"].astype(int)):
        x1, y1 = x0 + tile_w, y0 + tile_h
        if x0 > 0:
            seam_mask[y0:y1, max(0, x0 - border_width):min(W, x0 + border_width)] = True
        if x1 < W:
            seam_mask[y0:y1, max(0, x1 - border_width):min(W, x1 + border_width)] = True
        if y0 > 0:
            seam_mask[max(0, y0 - border_width):min(H, y0 + border_width), x0:x1] = True
        if y1 < H:
            seam_mask[max(0, y1 - border_width):min(H, y1 + border_width), x0:x1] = True

    if not seam_mask.any():
        return stitched_plane.copy()

    transition_mask = morphology.binary_dilation(seam_mask, morphology.disk(border_width))
    
    ys, xs = np.where(transition_mask)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    margin = max(median_size // 2, int(3 * sigma))
    y0, x0 = max(0, y0 - margin), max(0, x0 - margin)
    y1, x1 = min(H, y1 + margin), min(W, x1 + margin)

    crop = stitched_plane[y0:y1, x0:x1]
    seam_crop = seam_mask[y0:y1, x0:x1]
    trans_crop = transition_mask[y0:y1, x0:x1]

    median_crop = ndimage.median_filter(crop, size=median_size)
    gaussian_crop = ndimage.gaussian_filter(crop, sigma=sigma)

    out = stitched_plane.copy()
    out_crop = out[y0:y1, x0:x1]
    out_crop[seam_crop] = median_crop[seam_crop]
    blend_mask = trans_crop & ~seam_crop
    out_crop[blend_mask] = 0.5 * median_crop[blend_mask] + 0.5 * gaussian_crop[blend_mask]
    out[y0:y1, x0:x1] = out_crop
    
    return out

In [ ]:
def process_zplane_to_zarr(
    image_dir: str,
    store: KintsugiZarr,
    cycle: int,
    channel: int,
    zplane: int,
    rows: list,
    cols: list,
    pou: float,
    overlap_percentage: float,
    use_gpu: bool,
    smooth_borders: bool,
    stitch_model: dict = None
) -> dict:
    """
    Process a single z-plane: GPU BaSiC correction + stitching → zarr.
    
    Returns stitching model if this is the reference z-plane.
    """
    from skimage.io import imread_collection
    
    # Load tiles
    filename_pattern = f'1_000??_Z0{str(zplane).zfill(2)}_CH{str(channel)}.tif'
    im_raw = sorted(glob(os.path.join(image_dir, f'cyc{str(cycle).zfill(3)}', filename_pattern)), key=alphanumeric_key)
    im = imread_collection(im_raw)
    im_array_init = np.asarray(im)
    dtype_max = np.iinfo(im_array_init.dtype).max
    im_array = im_array_init.astype(np.float64) / dtype_max
    
    # GPU-accelerated BaSiC illumination correction
    flatfield, darkfield = KCorrectGPUFunc(
        im_array,
        if_darkfield=BASIC_IF_DARKFIELD,
        max_iterations=BASIC_MAX_ITERATIONS,
        optimization_tolerance=BASIC_OPTIMIZATION_TOLERANCE,
        max_reweight_iterations=BASIC_MAX_REWEIGHT_ITERATIONS,
        reweight_tolerance=BASIC_REWEIGHT_TOLERANCE,
        use_gpu=use_gpu
    )
    
    # Apply correction
    corrected = (im_array - darkfield) / flatfield
    corrected = np.clip(corrected, 0, 1)
    corrected = (corrected * dtype_max).astype(np.uint16)
    
    # Stitching
    compute_model = stitch_model is None
    
    if compute_model:
        result_df, _ = stitch_images(
            corrected, rows, cols,
            initial_ncc_threshold=0.078,
            overlap_percentage=overlap_percentage,
            pou=pou, use_gpu=use_gpu
        )
        stitch_model = {'result_df': result_df}
    else:
        result_df = stitch_model['result_df']
    
    # Normalize positions
    result_df = result_df.copy()
    result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
    result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()
    
    size_y, size_x = corrected.shape[1], corrected.shape[2]
    stitched_size = (
        int(result_df["y_pos2"].max() + size_y),
        int(result_df["x_pos2"].max() + size_x),
    )
    
    stitched = np.zeros(stitched_size, dtype=corrected.dtype)
    
    # Place tiles
    for i, row in result_df.iterrows():
        stitched[
            int(row["y_pos2"]):int(row["y_pos2"]) + size_y,
            int(row["x_pos2"]):int(row["x_pos2"]) + size_x,
        ] = corrected[i]
    
    # Border smoothing
    if smooth_borders:
        stitched = smooth_tile_borders_2d(stitched, result_df, (size_y, size_x))
    
    # Write to zarr
    store.write_stitched(cycle, channel, zplane, stitched, chunks=ZARR_CHUNKS)
    
    return stitch_model if compute_model else None


# Dask-parallel version for processing multiple z-planes concurrently (CPU mode)
def process_zplane_delayed_cpu(
    image_dir: str,
    zarr_path: str,
    cycle: int,
    channel: int,
    zplane: int,
    rows: list,
    cols: list,
    pou: float,
    overlap_percentage: float,
    smooth_borders: bool,
    stitch_model_dict: dict,
    basic_params: dict,
    zarr_chunks: tuple
) -> str:
    """
    Dask-compatible function for processing a single z-plane (CPU mode).
    
    Uses CPU for BaSiC to allow parallel execution across multiple workers.
    """
    from skimage.io import imread_collection
    import zarr
    
    # Load tiles
    filename_pattern = f'1_000??_Z0{str(zplane).zfill(2)}_CH{str(channel)}.tif'
    im_raw = sorted(glob(os.path.join(image_dir, f'cyc{str(cycle).zfill(3)}', filename_pattern)), key=alphanumeric_key)
    im = imread_collection(im_raw)
    im_array_init = np.asarray(im)
    dtype_max = np.iinfo(im_array_init.dtype).max
    im_array = im_array_init.astype(np.float64) / dtype_max
    
    # BaSiC correction (CPU-only to avoid GPU contention)
    flatfield, darkfield = KCorrectGPUFunc(
        im_array,
        if_darkfield=basic_params['if_darkfield'],
        max_iterations=basic_params['max_iterations'],
        optimization_tolerance=basic_params['optimization_tolerance'],
        max_reweight_iterations=basic_params['max_reweight_iterations'],
        reweight_tolerance=basic_params['reweight_tolerance'],
        use_gpu=False
    )
    
    # Apply correction
    corrected = (im_array - darkfield) / flatfield
    corrected = np.clip(corrected, 0, 1)
    corrected = (corrected * dtype_max).astype(np.uint16)
    
    # Use pre-computed stitch model
    result_df = stitch_model_dict['result_df'].copy()
    result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
    result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()
    
    size_y, size_x = corrected.shape[1], corrected.shape[2]
    stitched_size = (
        int(result_df["y_pos2"].max() + size_y),
        int(result_df["x_pos2"].max() + size_x),
    )
    
    stitched = np.zeros(stitched_size, dtype=corrected.dtype)
    
    # Place tiles
    for i, row in result_df.iterrows():
        stitched[
            int(row["y_pos2"]):int(row["y_pos2"]) + size_y,
            int(row["x_pos2"]):int(row["x_pos2"]) + size_x,
        ] = corrected[i]
    
    # Border smoothing
    if smooth_borders:
        stitched = smooth_tile_borders_2d(stitched, result_df, (size_y, size_x))
    
    # Write directly to zarr
    root = zarr.open(zarr_path, mode='a')
    cycle_grp = root[f'cyc{cycle:02d}']
    stitched_grp = cycle_grp['stitched']
    ch_grp = stitched_grp.require_group(f'CH{channel}')
    
    ch_grp.array(
        f'Z{zplane:02d}',
        stitched,
        chunks=zarr_chunks,
        dtype=stitched.dtype,
        overwrite=True
    )
    
    return f"cyc{cycle:02d}_CH{channel}_Z{zplane:02d}"


# Multi-GPU version using dask-cuda
def process_zplane_delayed_gpu(
    image_dir: str,
    zarr_path: str,
    cycle: int,
    channel: int,
    zplane: int,
    rows: list,
    cols: list,
    pou: float,
    overlap_percentage: float,
    smooth_borders: bool,
    stitch_model_dict: dict,
    basic_params: dict,
    zarr_chunks: tuple
) -> str:
    """
    Dask-CUDA compatible function for multi-GPU processing.
    
    Each worker automatically uses its assigned GPU via dask-cuda.
    """
    from skimage.io import imread_collection
    import zarr
    
    # Load tiles
    filename_pattern = f'1_000??_Z0{str(zplane).zfill(2)}_CH{str(channel)}.tif'
    im_raw = sorted(glob(os.path.join(image_dir, f'cyc{str(cycle).zfill(3)}', filename_pattern)), key=alphanumeric_key)
    im = imread_collection(im_raw)
    im_array_init = np.asarray(im)
    dtype_max = np.iinfo(im_array_init.dtype).max
    im_array = im_array_init.astype(np.float64) / dtype_max
    
    # BaSiC correction - GPU enabled (dask-cuda assigns GPU per worker)
    flatfield, darkfield = KCorrectGPUFunc(
        im_array,
        if_darkfield=basic_params['if_darkfield'],
        max_iterations=basic_params['max_iterations'],
        optimization_tolerance=basic_params['optimization_tolerance'],
        max_reweight_iterations=basic_params['max_reweight_iterations'],
        reweight_tolerance=basic_params['reweight_tolerance'],
        use_gpu=True  # Use GPU - dask-cuda handles device assignment
    )
    
    # Apply correction
    corrected = (im_array - darkfield) / flatfield
    corrected = np.clip(corrected, 0, 1)
    corrected = (corrected * dtype_max).astype(np.uint16)
    
    # Use pre-computed stitch model
    result_df = stitch_model_dict['result_df'].copy()
    result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
    result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()
    
    size_y, size_x = corrected.shape[1], corrected.shape[2]
    stitched_size = (
        int(result_df["y_pos2"].max() + size_y),
        int(result_df["x_pos2"].max() + size_x),
    )
    
    stitched = np.zeros(stitched_size, dtype=corrected.dtype)
    
    # Place tiles
    for i, row in result_df.iterrows():
        stitched[
            int(row["y_pos2"]):int(row["y_pos2"]) + size_y,
            int(row["x_pos2"]):int(row["x_pos2"]) + size_x,
        ] = corrected[i]
    
    # Border smoothing (CPU - minimal overhead)
    if smooth_borders:
        stitched = smooth_tile_borders_2d(stitched, result_df, (size_y, size_x))
    
    # Write directly to zarr
    root = zarr.open(zarr_path, mode='a')
    cycle_grp = root[f'cyc{cycle:02d}']
    stitched_grp = cycle_grp['stitched']
    ch_grp = stitched_grp.require_group(f'CH{channel}')
    
    ch_grp.array(
        f'Z{zplane:02d}',
        stitched,
        chunks=zarr_chunks,
        dtype=stitched.dtype,
        overwrite=True
    )
    
    return f"cyc{cycle:02d}_CH{channel}_Z{zplane:02d}"

In [ ]:
# Processing mode selection
# Options:
#   'sequential_gpu' - Process z-planes one at a time on GPU (best for single powerful GPU)
#   'parallel_cpu'   - Process z-planes in parallel on CPU workers (best for many CPU cores)
#   'multi_gpu'      - Process z-planes in parallel across multiple GPUs (best for multi-GPU systems)
PROCESSING_MODE = 'sequential_gpu'

# BaSiC parameters for parallel processing
basic_params = {
    'if_darkfield': BASIC_IF_DARKFIELD,
    'max_iterations': BASIC_MAX_ITERATIONS,
    'optimization_tolerance': BASIC_OPTIMIZATION_TOLERANCE,
    'max_reweight_iterations': BASIC_MAX_REWEIGHT_ITERATIONS,
    'reweight_tolerance': BASIC_REWEIGHT_TOLERANCE
}

print(f"Processing mode: {PROCESSING_MODE}")

# Process all cycles/channels/z-planes
total_ops = (end_cycle - start_cycle + 1) * (end_channel - start_channel + 1)

with tqdm(total=total_ops, desc='Correction & Stitching', colour="green") as pbar:
    for cycle in range(start_cycle, end_cycle + 1):
        for channel in range(start_channel, end_channel + 1):
            pbar.set_description(f'Cycle {cycle} CH{channel}')
            
            # Reference z-plane (middle) to get stitching model
            ref_zplane = n_zplanes // 2
            stitch_model = None
            
            if channel == 1:  # Compute model from CH1
                tqdm.write(f"Computing stitch model from cyc{cycle:02d} CH1 Z{ref_zplane:02d} (GPU)")
                stitch_model = process_zplane_to_zarr(
                    image_dir, store, cycle, channel, ref_zplane,
                    rows, cols, pou, overlap_percentage, use_gpu, smooth_borders
                )
                # Store model for other channels
                stored_model = stitch_model
            else:
                stitch_model = stored_model
            
            # Prepare list of z-planes to process
            zplanes_to_process = [z for z in range(1, n_zplanes + 1) if z != ref_zplane or channel != 1]
            
            if PROCESSING_MODE in ('parallel_cpu', 'multi_gpu'):
                # ============ PARALLEL MODES ============
                tqdm.write(f"Processing {len(zplanes_to_process)} z-planes in parallel ({PROCESSING_MODE})...")
                
                # Serialize stitch model for workers
                import pandas as pd
                stitch_model_serializable = {
                    'result_df': pd.DataFrame(stitch_model['result_df'].to_dict())
                }
                
                # Select appropriate delayed function
                delayed_func = process_zplane_delayed_gpu if PROCESSING_MODE == 'multi_gpu' else process_zplane_delayed_cpu
                
                # Create delayed tasks
                tasks = []
                for zplane in zplanes_to_process:
                    task = dask.delayed(delayed_func)(
                        image_dir=image_dir,
                        zarr_path=zarr_path,
                        cycle=cycle,
                        channel=channel,
                        zplane=zplane,
                        rows=rows,
                        cols=cols,
                        pou=pou,
                        overlap_percentage=overlap_percentage,
                        smooth_borders=smooth_borders,
                        stitch_model_dict=stitch_model_serializable,
                        basic_params=basic_params,
                        zarr_chunks=ZARR_CHUNKS
                    )
                    tasks.append(task)
                
                # Execute in parallel
                if PROCESSING_MODE == 'multi_gpu':
                    # For multi-GPU, submit to cluster
                    futures = client.compute(tasks)
                    results = client.gather(futures)
                else:
                    # For CPU parallel, use local scheduler with progress
                    from dask.diagnostics import ProgressBar
                    with ProgressBar():
                        results = dask.compute(*tasks)
                
                tqdm.write(f"  Completed: {len(results)} z-planes")
                
            else:
                # ============ SEQUENTIAL GPU MODE ============
                for zplane in tqdm(zplanes_to_process, desc=f'Z-planes', leave=False):
                    process_zplane_to_zarr(
                        image_dir, store, cycle, channel, zplane,
                        rows, cols, pou, overlap_percentage, use_gpu, smooth_borders,
                        stitch_model=stitch_model
                    )
            
            pbar.update(1)
            gc.collect()

print(f"\nStitching complete at {datetime.now()}")

---
## 6. Deconvolution

Read z-stacks from zarr, deconvolve, write back to zarr.

In [ ]:
from KDecon import KDecon as KDeconProcessor

# Deconvolution parameters
DECON_PARAMS = {
    'xy_vox': 377,          # XY pixel size (nm)
    'z_vox': 1500,          # Z pixel size (nm)
    'iterations': 25,       # Max LR iterations
    'mic_NA': 0.75,         # Numerical aperture
    'tissue_RI': 1.44,      # Refractive index
    'damping': 0,           # Noise damping
    'stop_criterion': 5.0,  # Stop if change < %
    'device': 'auto',       # GPU/CPU selection
}

# Channel wavelengths: {channel: (excitation_nm, emission_nm)}
WAVELENGTHS = {
    1: (358, 461),   # DAPI
    2: (753, 775),   # Cy7/AF750
    3: (560, 575),   # Cy3/AF555
    4: (648, 668)    # Cy5/AF647
}

In [ ]:
def deconvolve_channel_zarr(store: KintsugiZarr, cycle: int, channel: int):
    """
    Deconvolve a channel's z-stack from zarr and write back.
    """
    # Read z-stack as dask array
    stack_dask = store.read_stitched_dask(cycle, channel)
    
    # Compute to numpy (load into memory)
    # For very large data, process in chunks
    tqdm.write(f"Loading z-stack for cyc{cycle:02d} CH{channel}...")
    stack = stack_dask.compute()
    
    # Transpose to (x, y, z) for KDecon
    stack = stack.transpose(2, 1, 0)  # (z,y,x) -> (x,y,z)
    
    # Get wavelengths
    lambda_ex, lambda_em = WAVELENGTHS.get(channel, (560, 575))
    
    # Create deconvolver
    decon = KDeconProcessor(
        dxy=DECON_PARAMS['xy_vox'],
        dz=DECON_PARAMS['z_vox'],
        NA=DECON_PARAMS['mic_NA'],
        rf=DECON_PARAMS['tissue_RI'],
        lambda_ex=lambda_ex,
        lambda_em=lambda_em,
        iterations=DECON_PARAMS['iterations'],
        damping=DECON_PARAMS['damping'],
        stop_criterion=DECON_PARAMS['stop_criterion'],
        device=DECON_PARAMS['device'],
        verbose=True
    )
    
    # Process
    tqdm.write(f"Deconvolving cyc{cycle:02d} CH{channel}...")
    result = decon.process_array(stack.astype(np.float32))
    
    # Transpose back to (z, y, x)
    result = result.transpose(2, 1, 0)
    
    # Write to zarr
    store.write_deconvolved(cycle, channel, result.astype(np.uint16))
    tqdm.write(f"Saved deconvolved cyc{cycle:02d} CH{channel} to zarr")
    
    del stack, result
    gc.collect()

In [ ]:
# Run deconvolution (skip DAPI channel 1)
decon_start_channel = 2  # Skip DAPI

total_ops = (end_cycle - start_cycle + 1) * (end_channel - decon_start_channel + 1)

with tqdm(total=total_ops, desc='Deconvolution', colour="blue") as pbar:
    for cycle in range(start_cycle, end_cycle + 1):
        for channel in range(decon_start_channel, end_channel + 1):
            pbar.set_description(f'Decon Cyc{cycle} CH{channel}')
            
            try:
                deconvolve_channel_zarr(store, cycle, channel)
            except Exception as e:
                tqdm.write(f"Error deconvolving cyc{cycle} CH{channel}: {e}")
            
            pbar.update(1)
            gc.collect()

print(f"\nDeconvolution complete at {datetime.now()}")

---
## 7. Extended Depth of Focus (EDF)

Project z-stacks to 2D using variance-based focus selection.

In [ ]:
# EDF parameters
EDF_PARAMS = {
    'radius_x': 5,
    'radius_y': 5,
    'sigma': 20.0,
    'z_start': 1,
    'z_end': 15,
    'tiles': (1, 2),  # Tiling for large images
    'backend': 'auto'  # 'cupy' or 'numpy'
}

# Initialize EDF processor
edf_processor = EDFProcessor(backend=EDF_PARAMS['backend'], method='variance')
print(f"EDF using backend: {edf_processor.backend}")

In [ ]:
def process_edf_zarr(store: KintsugiZarr, cycle: int, channel: int, use_deconvolved: bool = True):
    """
    Apply EDF to z-stack and write result to zarr.
    """
    # Read z-stack
    if use_deconvolved and channel > 1:  # DAPI not deconvolved
        stack_dask = store.read_deconvolved_dask(cycle, channel)
    else:
        stack_dask = store.read_stitched_dask(cycle, channel)
    
    tqdm.write(f"Loading stack for EDF cyc{cycle:02d} CH{channel}...")
    stack = stack_dask.compute()
    
    # Apply EDF
    tqdm.write(f"Processing EDF cyc{cycle:02d} CH{channel}...")
    result = edf_processor.process(
        stack,
        radius_x=EDF_PARAMS['radius_x'],
        radius_y=EDF_PARAMS['radius_y'],
        sigma=EDF_PARAMS['sigma'],
        z_start=EDF_PARAMS['z_start'],
        z_end=EDF_PARAMS['z_end'],
        tiles=EDF_PARAMS['tiles']
    )
    
    # Get channel name
    ch_name = channel_name_dict.get(cycle, [''] * 4)[channel - 1]
    
    # Write to zarr
    store.write_edf(cycle, channel, result, channel_name=ch_name)
    tqdm.write(f"Saved EDF {ch_name} to zarr")
    
    del stack, result
    gc.collect()

In [ ]:
# Run EDF processing
edf_start_cycle = 1
edf_end_cycle = end_cycle

total_ops = (edf_end_cycle - edf_start_cycle + 1) * (end_channel - start_channel + 1)

with tqdm(total=total_ops, desc='EDF Processing', colour="purple") as pbar:
    for cycle in range(edf_start_cycle, edf_end_cycle + 1):
        for channel in range(start_channel, end_channel + 1):
            pbar.set_description(f'EDF Cyc{cycle} CH{channel}')
            
            try:
                process_edf_zarr(store, cycle, channel, use_deconvolved=True)
            except Exception as e:
                tqdm.write(f"Error EDF cyc{cycle} CH{channel}: {e}")
            
            pbar.update(1)
            gc.collect()

print(f"\nEDF complete at {datetime.now()}")

---
## 8. Export to OME-NGFF for Registration

Write EDF results as proper OME-NGFF format for VALIS registration.

In [ ]:
# Export each cycle's EDF as OME-NGFF
pixel_size_um = 0.377

for cycle in range(start_cycle, end_cycle + 1):
    # Get all channels for this cycle
    edf_stack = store.get_all_edf_dask(cycle)
    edf_data = edf_stack.compute()
    
    ch_names = channel_name_dict.get(cycle, [f'CH{i}' for i in range(1, 5)])
    
    # Write OME-NGFF with pyramid
    store.write_ome_zarr(
        cycle=cycle,
        data=edf_data,
        axes="cyx",
        channel_names=ch_names,
        pixel_size_um=pixel_size_um,
        generate_pyramid=True,
        pyramid_levels=4
    )
    
    print(f"Exported cycle {cycle} to OME-NGFF")

print(f"\nOME-NGFF export complete. Ready for registration.")

---
## 9. Cleanup and Summary

In [ ]:
# Close Dask cluster
client.close()
cluster.close()

# Print summary
print("\n" + "="*60)
print("Processing Summary")
print("="*60)

for cycle in store.list_cycles():
    info = store.get_cycle_info(cycle)
    print(f"\nCycle {cycle}:")
    for stage in info['stages']:
        contents = info.get(f'{stage}_contents', [])
        print(f"  {stage}: {len(contents)} items")

# Close zarr store
store.close()

print(f"\n" + "="*60)
print(f"Pipeline complete at {datetime.now()}")
print(f"Output: {zarr_path}")
print("="*60)

---
## Notes

### OME-Zarr Structure

```
experiment.zarr/
├── cyc01/
│   ├── raw/           # Not used in this notebook
│   ├── corrected/     # BaSiC corrected (intermediate)
│   ├── stitched/      # Stitched mosaics (z, y, x)
│   │   ├── CH1/
│   │   │   ├── Z01
│   │   │   ├── Z02
│   │   │   └── ...
│   │   ├── CH2/
│   │   └── ...
│   ├── deconvolved/   # Deconvolved stacks
│   └── edf/           # 2D projections
├── cyc02/
│   └── ...
├── ome_ngff/          # OME-NGFF export for registration
│   ├── cyc01.zarr
│   └── ...
└── registered/        # Final registered output
```

### Processing Modes

| Mode | Best For | Description |
|------|----------|-------------|
| `sequential_gpu` | Single powerful GPU | Process z-planes one at a time using GPU BaSiC. Best throughput when GPU is fast. |
| `parallel_cpu` | Many CPU cores, limited GPU | Process z-planes in parallel on CPU workers. Uses Dask LocalCluster. |
| `multi_gpu` | Multi-GPU systems | Each GPU processes z-planes independently via dask-cuda. Requires `dask-cuda`. |

### Cluster Modes

| Mode | Workers | Description |
|------|---------|-------------|
| `local` | CPU workers | Standard Dask LocalCluster for CPU parallel processing |
| `gpu` | 2 workers | Reduced workers for single GPU to avoid memory contention |
| `multi_gpu` | 1 per GPU | dask-cuda LocalCUDACluster with automatic GPU assignment |

### GPU BaSiC Performance

The `kcorrect_gpu` module provides 10-50x speedup over CPU for BaSiC illumination correction:

- **GPU acceleration**: Uses CuPy for all matrix operations
- **DCT/IDCT**: GPU-accelerated via `cupyx.scipy.fft`
- **Auto-fallback**: Falls back to NumPy/SciPy if CuPy unavailable
- **Memory efficient**: Working size downsampling (128x128 default)

### Performance Tips

1. **Chunk size**: Adjust `ZARR_CHUNKS` based on your GPU memory (default: 1024x1024)
2. **Workers**: 
   - For `parallel_cpu`: Match to CPU cores
   - For `multi_gpu`: One worker per GPU (automatic)
3. **Memory limit**: Increase in LocalCluster for large images
4. **Multi-GPU requirements**:
   - Install: `conda install -c rapidsai dask-cuda`
   - RMM pool: Faster memory allocations
   - NVLink: Enable for GPU-to-GPU transfers

### Installation for Multi-GPU

```bash
# Install dask-cuda from RAPIDS
conda install -c rapidsai -c conda-forge -c nvidia dask-cuda cudatoolkit=12.0

# Or with pip (requires CUDA toolkit)
pip install dask-cuda
```